In [1]:
# Asegurate de que estas con el Kernel de Python 3.10, mediapipe no funciona con el 11 ( TODO ESTO ES DESDE LA TERMINAL)
# py -3.10 -m ipykernel install --user --name "python310" --display-name "Python 3.10"
# py -3.10 -m pip install --upgrade pip
# py -3.10 -m pip install mediapipe -- comprobar q se ha descargado bien : py -3.10 -m pip show mediapipe opencv-python pyttsx3 numpy

In [1]:
# Comprobacion final de que todo esta instalado
import mediapipe as mp
import cv2
import pyttsx3
import numpy as np

print("All packages imported successfully!")

All packages imported successfully!


In [8]:
# La camara tarda en abrirse

# CON FEEDBACK

In [2]:
# Importar librerías
import time
import os
import pyttsx3
import threading
import numpy as np
import mediapipe as mp
from queue import Queue
import cv2
import time
import os
import pyttsx3
import threading
import numpy as np
import mediapipe as mp
from queue import Queue

## Inicializar MediaPipe

In [3]:
# Inicializar MediaPipe
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

COLOR_IZQUIERDA = (0, 0, 255)
COLOR_DERECHA = (0, 255, 0)

landmarks_izquierda = {
    mp_pose.PoseLandmark.LEFT_SHOULDER,
    mp_pose.PoseLandmark.LEFT_ELBOW,
    mp_pose.PoseLandmark.LEFT_WRIST,
    mp_pose.PoseLandmark.LEFT_HIP,
    mp_pose.PoseLandmark.LEFT_KNEE,
    mp_pose.PoseLandmark.LEFT_ANKLE,
    mp_pose.PoseLandmark.LEFT_HEEL,
    mp_pose.PoseLandmark.LEFT_FOOT_INDEX,
}
landmarks_derecha = {
    mp_pose.PoseLandmark.RIGHT_SHOULDER,
    mp_pose.PoseLandmark.RIGHT_ELBOW,
    mp_pose.PoseLandmark.RIGHT_WRIST,
    mp_pose.PoseLandmark.RIGHT_HIP,
    mp_pose.PoseLandmark.RIGHT_KNEE,
    mp_pose.PoseLandmark.RIGHT_ANKLE,
    mp_pose.PoseLandmark.RIGHT_HEEL,
    mp_pose.PoseLandmark.RIGHT_FOOT_INDEX,
}

## Calcular ángulos

In [4]:
def calcular_angulo(p1, p2, p3):
    # Si no detecta algún punto se devuelve "None"
    if p1 is None or p2 is None or p3 is None:
        return None
        
    a = np.array([p1.x, p1.y])
    b = np.array([p2.x, p2.y])
    c = np.array([p3.x, p3.y])
    
    ba = a - b
    bc = c - b

    if np.linalg.norm(ba) == 0 or np.linalg.norm(bc) == 0:
        return None
    
    cos_ang = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angulo = np.arccos(np.clip(cos_ang, -1.0, 1.0))
    
    return np.degrees(angulo)

## Mensajes a reproducir

In [5]:
# Mensajes y espacio tras reproducción
mensajes = [
    # Saludo
    [("Buenos días, le voy a explicar un par de ejercicios que quiero que haga esta mañana.", 2)],
    # Ejercicio 1
    [("El primer ejercicio consiste en levantar y bajar la pierna, quiero que haga 6 repeticiones con cada pierna. Empiece con la pierna izquierda.", 30)],
    [("Ahora cambie a la pierna derecha", 30)],
    # Ejercicio 2
    [("Para el siguiente ejercicio quiero que levante la pierna y la mantenga estirada y arriba lo máximo que pueda. Empiece con la pierna izquierda.", 30)],
    [("Ahora cambie a la pierna derecha", 30)],
    # Ejercicio 3
    [("En este último ejercicio tiene que estirar la pierna hacia arriba e intente apuntar con el dedo gordo del pie hacia su nariz, quiero que haga 6 repeticiones con cada pierna. Empiece con la pierna izquierda.", 30)],
    [("Ahora cambie a la pierna derecha", 30)],
    # Fin
    [("Con esto hemos terminado los ejercicios por hoy. Muchas gracias.", 0)]
]

## Función principal

In [7]:
# Evento para controlar el audio (para detener la reproducción de mensajes de audio) 
stop_audio = threading.Event()

# Cola para almacenar mensajes
cola_voz = Queue()
etapa_audio = -1  # -1 antes de comenzar

# Función para reproducir los mensajes
def reproducir_mensajes():
    global etapa_audio
    # Inicializar el motor de texto a voz
    engine = pyttsx3.init()
    engine.setProperty('rate', 150) # velocidad
    engine.setProperty('volume', 0.8) # volumen

    for i, grupo in enumerate(mensajes):
        if stop_audio.is_set():
            break
        etapa_audio = i # actualizar etapa
        principal, pausa_total = grupo[0]
        cola_voz.put(principal) # poner mensaje a la cola

        intermedios = grupo[1:]
        enviados = set()
        start = time.time()

        while time.time() - start < pausa_total:
            if stop_audio.is_set():
                break
            elapsed = int(time.time() - start)
            for msg, delay in intermedios:
                if elapsed >= delay and msg not in enviados:
                    cola_voz.put(msg)
                    enviados.add(msg)
            time.sleep(1)

# Función para el hilo de mensajes
def hilo_voz():
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 1.0)
    while not stop_audio.is_set():
        try:
            msg = cola_voz.get(timeout=1) # obtener el mensaje de la cola
            engine.say(msg) # reproducir
            engine.runAndWait()
            time.sleep(0.5)  # pequeña pausa entre mensajes
        except:
            continue

# Función principal para grabar video y procesar poses
def grabar_con_pose(duracion_segundos=210):
    print("Presiona Enter para iniciar la grabación.")
    input()
    print("Grabación iniciada. Pulsa 'espacio' o 'q' para detener.")

    # Abrir la cámara
    cap = cv2.VideoCapture(0) # 0 cámara ordenador, 1 webcam
    if not cap.isOpened():
        print("No se pudo acceder a la cámara.")
        return

    # Dimensiones del vídeo
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = 20

    # Directorio de salida
    output_path = os.path.join(os.path.expanduser("~"), "Escritorio", "VIDEOS")
    os.makedirs(output_path, exist_ok=True)
    num = 1
    while True:
        filename = f"video_full_{num}.avi"
        video_path = os.path.join(output_path, filename)
        if not os.path.exists(video_path):
            break
        num += 1
    out = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*'XVID'), fps, (frame_width, frame_height))

    # Inicializar los hilos 
    stop_audio.clear()
    threading.Thread(target=reproducir_mensajes).start()
    threading.Thread(target=hilo_voz).start()

    # Variables de retroalimentación 
    ultimo_alerta = [0, 0, 0, 0, 0, 0]
    intervalo_alerta = [10, 10, 10, 10, 10, 10]
    conteo_alertas = [0, 0, 0, 0, 0, 0]
    max_alertas = [3, 3, 3, 3, 3, 3]

    # Variables de mensajes de ánimo
    mensaje_aguante_reproducido_i = False
    mensaje_aguante_reproducido_d = False
    mensaje_siga_reproducido_i = False
    mensaje_siga_reproducido_d = False

    #variables de inicio de etapas
    inicio_etapa_1 = None
    inicio_etapa_2 = None
    inicio_etapa_3 = None
    inicio_etapa_4 = None
    inicio_etapa_5 = None
    inicio_etapa_6 = None


    # MediaPipe para detección de poses
    with mp_pose.Pose(
        static_image_mode=False, 
        model_complexity=2, 
        enable_segmentation=False, 
        min_detection_confidence=0.5) as pose:
        
        start = time.time()

        while True:
            
            # Registrar el tiempo de inicio de cada etapa cuando entremos por primera vez
            if etapa_audio == 1 and inicio_etapa_1 is None:
                inicio_etapa_1 = time.time()
            if etapa_audio == 2 and inicio_etapa_2 is None:
                inicio_etapa_2 = time.time()
            if etapa_audio == 3 and inicio_etapa_3 is None:
                inicio_etapa_3 = time.time()
            if etapa_audio == 4 and inicio_etapa_4 is None:
                inicio_etapa_4 = time.time()
            if etapa_audio == 5 and inicio_etapa_5 is None:
                inicio_etapa_5 = time.time()
            if etapa_audio == 6 and inicio_etapa_6 is None:
                inicio_etapa_6 = time.time()

            if time.time() - start > duracion_segundos:
                print("Tiempo máximo alcanzado.")
                break

            ret, frame = cap.read()
            if not ret:
                print("Error leyendo el frame.")
                break

            annotated = frame.copy() # copiar para anotar sobre el frame
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # convertir a RGB
            results = pose.process(rgb) # procesar

            # Analizar pose
            if results.pose_landmarks:
                mp_drawing.draw_landmarks(
                    annotated,
                    results.pose_landmarks,
                    mp_pose.POSE_CONNECTIONS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
                )
           
               
                for idx, lm in enumerate(results.pose_landmarks.landmark):
                    x = int(lm.x * frame_width)
                    y = int(lm.y * frame_height)
                    if idx in [l.value for l in landmarks_izquierda]:
                        cv2.circle(annotated, (x, y), 4, COLOR_IZQUIERDA, -1)
                    elif idx in [l.value for l in landmarks_derecha]:
                        cv2.circle(annotated, (x, y), 4, COLOR_DERECHA, -1)

                # Calcular ángulos
                for lado, color in zip(["izquierda", "derecha"], [COLOR_IZQUIERDA, COLOR_DERECHA]):
                    try:
                        cadera = results.pose_landmarks.landmark[
                            mp_pose.PoseLandmark.LEFT_HIP if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_HIP]
                        rodilla = results.pose_landmarks.landmark[
                            mp_pose.PoseLandmark.LEFT_KNEE if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_KNEE]
                        tobillo = results.pose_landmarks.landmark[
                            mp_pose.PoseLandmark.LEFT_ANKLE if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_ANKLE]
                        pie = results.pose_landmarks.landmark[
                            mp_pose.PoseLandmark.LEFT_FOOT_INDEX if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_FOOT_INDEX]
                
                        # Ángulo rodilla 
                        angulo_rodilla = calcular_angulo(cadera, rodilla, tobillo)
                        x_rodilla = int(rodilla.x * frame_width)
                        y_rodilla = int(rodilla.y * frame_height)
                        cv2.putText(annotated, f"R:{int(angulo_rodilla)}deg", (x_rodilla - 20, y_rodilla - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

                        if lado == "izquierda":
                            angulo_rodilla_izq = angulo_rodilla
                        else:
                            angulo_rodilla_der = angulo_rodilla
                
                        # Ángulo tobillo 
                        angulo_tobillo = calcular_angulo(rodilla, tobillo, pie)
                        x_tobillo = int(tobillo.x * frame_width)
                        y_tobillo = int(tobillo.y * frame_height)
                        cv2.putText(annotated, f"T:{int(angulo_tobillo)}deg", (x_tobillo - 20, y_tobillo - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

                        if lado == "izquierda":
                            angulo_tobillo_izq = angulo_tobillo
                        else:
                            angulo_tobillo_der = angulo_tobillo
               
                        # Determinar la pierna activa
                        margen = 20
                        pierna_activa = None
                        if angulo_rodilla_izq is not None and angulo_rodilla_der is not None:
                            if abs(angulo_rodilla_izq - angulo_rodilla_der) > margen:
                                if angulo_rodilla_izq > angulo_rodilla_der:
                                    pierna_activa = "izquierda"
                                else:
                                    pierna_activa = "derecha"
                            else:
                                pierna_activa = "iguales"
                                
                        
                        # Feedback Ejercicio 1  
                        # Pierna izquierda
                        if (etapa_audio == 1 and pierna_activa == "derecha" and
                            angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                            time.time() - inicio_etapa_1 >= 5 and  # aquí controlamos los 5 segundos
                            time.time() - ultimo_alerta[0] > intervalo_alerta[0] and 
                            conteo_alertas[0] < max_alertas[0]):
                            cola_voz.put("Levante la pierna izquierda.")
                            ultimo_alerta[0] = time.time()
                            conteo_alertas[0] += 1

                        # Pierna derecha
                        if (etapa_audio == 2 and pierna_activa == "izquierda" and
                            angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                            time.time() - inicio_etapa_2 >= 5 and  # aquí controlamos los 5 segundos
                            time.time() - ultimo_alerta[1] > intervalo_alerta[1] and 
                            conteo_alertas[1] < max_alertas[1]):
                            cola_voz.put("Levante la pierna derecha.")
                            ultimo_alerta[1] = time.time()
                            conteo_alertas[1] += 1      

                        # Feedback Ejercicio 2
                        # Pierna izquierda
                        if etapa_audio == 3:
                            if angulo_rodilla_izq is not None and angulo_rodilla_der is not None: 
                                ambos_bajo_120 = angulo_rodilla_izq < 120 and angulo_rodilla_der < 120
                                alguno_arriba_120 = angulo_rodilla_izq >= 120 or angulo_rodilla_der >= 120
                                                    
                            if (pierna_activa == "derecha" and
                                angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                time.time() - ultimo_alerta[2] > intervalo_alerta[2] and 
                                conteo_alertas[2] < max_alertas[2]):
                                cola_voz.put("Levante la pierna izquierda.")
                                ultimo_alerta[2] = time.time()
                                conteo_alertas[2] += 1
                            
                            if (ambos_bajo_120 and
                                time.time() - ultimo_alerta[2] > intervalo_alerta[2] and 
                                conteo_alertas[2] < max_alertas[2]):
                                    cola_voz.put("Suba un poco más la pierna")
                                    ultimo_alerta[2] = time.time()
                                    conteo_alertas[2] += 1
                            
                            elif (alguno_arriba_120 and not mensaje_aguante_reproducido_i and time.time() - inicio_etapa_3 >= 15):
                                mensaje = "¡Aguante un poco más!"
                                cola_voz.put(mensaje)
                                mensaje_aguante_reproducido_i = True
                            
                                tiempo_actual = time.time() - start
                                ejercicio_actual = f"Ejercicio 2"
                                
                                writer.writerow([
                                    round(tiempo_actual, 2),
                                    ejercicio_actual,
                                    round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                    round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                    round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                    round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                    mensaje
                                ])

                        # Pierna derecha
                        if etapa_audio == 4:
                            if angulo_rodilla_izq is not None and angulo_rodilla_der is not None: 
                                ambos_bajo_120 = angulo_rodilla_izq < 120 and angulo_rodilla_der < 120
                                alguno_arriba_120 = angulo_rodilla_izq >= 120 or angulo_rodilla_der >= 120
                                                    
                            if (pierna_activa == "izquierda" and
                                angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                time.time() - ultimo_alerta[3] > intervalo_alerta[3] and 
                                conteo_alertas[3] < max_alertas[3]):
                                cola_voz.put("Levante la pierna derecha.")
                                ultimo_alerta[3] = time.time()
                                conteo_alertas[3] += 1
                            
                            if (ambos_bajo_120 and
                                time.time() - ultimo_alerta[3] > intervalo_alerta[3] and 
                                conteo_alertas[3] < max_alertas[3]):
                                    cola_voz.put("Suba un poco más la pierna")
                                    ultimo_alerta[3] = time.time()
                                    conteo_alertas[3] += 1
                            
                            elif (alguno_arriba_120 and not mensaje_aguante_reproducido_d and time.time() - inicio_etapa_4 >= 15):
                                mensaje = "¡Aguante un poco más!"
                                cola_voz.put(mensaje)
                                mensaje_aguante_reproducido_d = True
                            
                                tiempo_actual = time.time() - start
                                ejercicio_actual = f"Ejercicio 2"
                                
                                writer.writerow([
                                    round(tiempo_actual, 2),
                                    ejercicio_actual,
                                    round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                    round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                    round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                    round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                    mensaje
                                ])


                        # Feedback ejercicio 3
                        # Pierna izquierda
                        if etapa_audio == 5:
                            if angulo_rodilla_izq is not None and angulo_rodilla_der is not None:
                                ambas_bajo_110 = angulo_rodilla_izq < 110 and angulo_rodilla_der < 110
                                alguna_arriba_110 = angulo_rodilla_izq >= 110 or angulo_rodilla_der >= 110
                        
                            if (pierna_activa == "derecha" and
                                angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                time.time() - ultimo_alerta[4] > intervalo_alerta[4] and 
                                conteo_alertas[4] < max_alertas[4]):
                                cola_voz.put("Levante la pierna izquierda.")
                                ultimo_alerta[4] = time.time()
                                conteo_alertas[4] += 1
                        
                            if ambas_bajo_110:
                                if (time.time() - ultimo_alerta[4] > intervalo_alerta[4] and 
                                    conteo_alertas[4] < max_alertas[4]):
                                    cola_voz.put("No baje la pierna")
                                    ultimo_alerta[4] = time.time()
                                    conteo_alertas[4] += 1
                        
                            elif (alguna_arriba_110 and not mensaje_siga_reproducido_i and time.time() - inicio_etapa_5 >= 15):  # Aquí controlamos el retraso de 15s
                                cola_voz.put("¡Siga así, ya casi está!")
                                mensaje_siga_reproducido_i = True

                        # Pierna derecha
                        if etapa_audio == 6:
                            if angulo_rodilla_izq is not None and angulo_rodilla_der is not None:
                                ambas_bajo_110 = angulo_rodilla_izq < 110 and angulo_rodilla_der < 110
                                alguna_arriba_110 = angulo_rodilla_izq >= 110 or angulo_rodilla_der >= 110
                        
                            if (pierna_activa == "izquierda" and
                                angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                time.time() - ultimo_alerta[5] > intervalo_alerta[5] and 
                                conteo_alertas[5] < max_alertas[5]):
                                cola_voz.put("Levante la pierna derecha.")
                                ultimo_alerta[5] = time.time()
                                conteo_alertas[5] += 1
                        
                            if ambas_bajo_110:
                                if (time.time() - ultimo_alerta[5] > intervalo_alerta[5] and 
                                    conteo_alertas[5] < max_alertas[5]):
                                    cola_voz.put("No baje la pierna")
                                    ultimo_alerta[5] = time.time()
                                    conteo_alertas[5] += 1
                        
                            elif (alguna_arriba_110 and not mensaje_siga_reproducido_d and 
                                  time.time() - inicio_etapa_6 >= 15):  # Aquí controlamos el retraso de 15s
                                cola_voz.put("¡Siga así, ya casi está!")
                                mensaje_siga_reproducido_d = True
                    
                    except:
                        pass
                            


            # Guardar frame y mostrarlo en la vetana
            out.write(annotated)
            cv2.imshow('Pose en tiempo real', annotated)
          
            key = cv2.waitKey(1) & 0xFF
            if key == 32 or key == ord('q'):
                print("Grabación detenida por el usuario.")
                break

    stop_audio.set()
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print(f"Video guardado en: {filename}")

# Ejecutar
grabar_con_pose()

Presiona Enter para iniciar la grabación.


Grabación iniciada. Pulsa 'espacio' o 'q' para detener.
Grabación detenida por el usuario.
Video guardado en: video_full_2.avi


## GUARDAR CSV

In [13]:
import csv

# Evento para controlar el audio (para detener la reproducción de mensajes de audio) 
stop_audio = threading.Event()

# Cola para almacenar mensajes
cola_voz = Queue()
etapa_audio = -1  # -1 antes de comenzar

# Función para reproducir los mensajes
def reproducir_mensajes():
    global etapa_audio
    # Inicializar el motor de texto a voz
    engine = pyttsx3.init()
    engine.setProperty('rate', 150) # velocidad
    engine.setProperty('volume', 0.8) # volumen

    for i, grupo in enumerate(mensajes):
        if stop_audio.is_set():
            break
        etapa_audio = i # actualizar etapa
        principal, pausa_total = grupo[0]
        cola_voz.put(principal) # poner mensaje a la cola

        intermedios = grupo[1:]
        enviados = set()
        start = time.time()

        while time.time() - start < pausa_total:
            if stop_audio.is_set():
                break
            elapsed = int(time.time() - start)
            for msg, delay in intermedios:
                if elapsed >= delay and msg not in enviados:
                    cola_voz.put(msg)
                    enviados.add(msg)
            time.sleep(1)

# Función para el hilo de mensajes
def hilo_voz():
    engine = pyttsx3.init()
    engine.setProperty('rate', 150)
    engine.setProperty('volume', 1.0)
    while not stop_audio.is_set():
        try:
            msg = cola_voz.get(timeout=1) # obtener el mensaje de la cola
            engine.say(msg) # reproducir
            engine.runAndWait()
            time.sleep(0.5)  # pequeña pausa entre mensajes
        except:
            continue

# Función principal para grabar video y procesar poses
def grabar_con_pose(duracion_segundos=210):
    print("Presiona Enter para iniciar la grabación.")
    input()
    print("Grabación iniciada. Pulsa 'espacio' o 'q' para detener.")

    # Abrir la cámara
    cap = cv2.VideoCapture(0) # 0 cámara ordenador, 1 webcam
    if not cap.isOpened():
        print("No se pudo acceder a la cámara.")
        return

    # Dimensiones del vídeo
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = 20

    # Directorio de salida
    output_path = os.path.join(os.path.expanduser("~"), "Escritorio", "VIDEOS")
    os.makedirs(output_path, exist_ok=True)
    num = 1
    while True:
        filename = f"video_full_{num}.avi"
        video_path = os.path.join(output_path, filename)
        if not os.path.exists(video_path):
            break
        num += 1
    out = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*'XVID'), fps, (frame_width, frame_height))
    csv_path = os.path.join(output_path, "datos_pose.csv")

    with open(csv_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([
        "Tiempo", "Ejercicio", 
        "Ángulo Rodilla Izquierda", "Ángulo Rodilla Derecha", 
        "Ángulo Tobillo Izquierdo", "Ángulo Tobillo Derecho", 
        "Detalle Alerta"
    ])
        # Inicializar los hilos 
        stop_audio.clear()
        threading.Thread(target=reproducir_mensajes).start()
        threading.Thread(target=hilo_voz).start()
    
        # Variables de retroalimentación 
        ultimo_alerta = [0, 0, 0, 0, 0, 0]
        intervalo_alerta = [10, 10, 10, 10, 10, 10]
        conteo_alertas = [0, 0, 0, 0, 0, 0]
        max_alertas = [3, 3, 3, 3, 3, 3]
    
        # Variables de mensajes de ánimo
        mensaje_aguante_reproducido_i = False
        mensaje_aguante_reproducido_d = False
        mensaje_siga_reproducido_i = False
        mensaje_siga_reproducido_d = False

        #variables de inicio de etapas
        inicio_etapa_1 = None
        inicio_etapa_2 = None
        inicio_etapa_3 = None
        inicio_etapa_4 = None
        inicio_etapa_5 = None
        inicio_etapa_6 = None
    
        # MediaPipe para detección de poses
        with mp_pose.Pose(
            static_image_mode=False, 
            model_complexity=2, 
            enable_segmentation=False, 
            min_detection_confidence=0.5) as pose:
            
            start = time.time()
    
            while True:

                # Registrar el tiempo de inicio de cada etapa cuando entremos por primera vez
                if etapa_audio == 1 and inicio_etapa_1 is None:
                    inicio_etapa_1 = time.time()
                if etapa_audio == 2 and inicio_etapa_2 is None:
                    inicio_etapa_2 = time.time()
                if etapa_audio == 3 and inicio_etapa_3 is None:
                    inicio_etapa_3 = time.time()
                if etapa_audio == 4 and inicio_etapa_4 is None:
                    inicio_etapa_4 = time.time()
                if etapa_audio == 5 and inicio_etapa_5 is None:
                    inicio_etapa_5 = time.time()
                if etapa_audio == 6 and inicio_etapa_6 is None:
                    inicio_etapa_6 = time.time()
                
                if time.time() - start > duracion_segundos:
                    print("Tiempo máximo alcanzado.")
                    break
    
                ret, frame = cap.read()
                if not ret:
                    print("Error leyendo el frame.")
                    break
    
                annotated = frame.copy() # copiar para anotar sobre el frame
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # convertir a RGB
                results = pose.process(rgb) # procesar
    
                # Analizar pose
                if results.pose_landmarks:
                    mp_drawing.draw_landmarks(
                        annotated,
                        results.pose_landmarks,
                        mp_pose.POSE_CONNECTIONS,
                        landmark_drawing_spec=None,
                        connection_drawing_spec=mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2)
                    )
               
                   
                    for idx, lm in enumerate(results.pose_landmarks.landmark):
                        x = int(lm.x * frame_width)
                        y = int(lm.y * frame_height)
                        if idx in [l.value for l in landmarks_izquierda]:
                            cv2.circle(annotated, (x, y), 4, COLOR_IZQUIERDA, -1)
                        elif idx in [l.value for l in landmarks_derecha]:
                            cv2.circle(annotated, (x, y), 4, COLOR_DERECHA, -1)
    
                    # Calcular ángulos
                    for lado, color in zip(["izquierda", "derecha"], [COLOR_IZQUIERDA, COLOR_DERECHA]):
                        try:
                            cadera = results.pose_landmarks.landmark[
                                mp_pose.PoseLandmark.LEFT_HIP if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_HIP]
                            rodilla = results.pose_landmarks.landmark[
                                mp_pose.PoseLandmark.LEFT_KNEE if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_KNEE]
                            tobillo = results.pose_landmarks.landmark[
                                mp_pose.PoseLandmark.LEFT_ANKLE if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_ANKLE]
                            pie = results.pose_landmarks.landmark[
                                mp_pose.PoseLandmark.LEFT_FOOT_INDEX if lado == "izquierda" else mp_pose.PoseLandmark.RIGHT_FOOT_INDEX]
                    
                            # Ángulo rodilla 
                            angulo_rodilla = calcular_angulo(cadera, rodilla, tobillo)
                            x_rodilla = int(rodilla.x * frame_width)
                            y_rodilla = int(rodilla.y * frame_height)
                            cv2.putText(annotated, f"{int(angulo_rodilla)}deg", (x_rodilla - 20, y_rodilla - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
                            if lado == "izquierda":
                                angulo_rodilla_izq = angulo_rodilla
                            else:
                                angulo_rodilla_der = angulo_rodilla
                    
                            # Ángulo tobillo 
                            angulo_tobillo = calcular_angulo(rodilla, tobillo, pie)
                            x_tobillo = int(tobillo.x * frame_width)
                            y_tobillo = int(tobillo.y * frame_height)
                            cv2.putText(annotated, f"{int(angulo_tobillo)}deg", (x_tobillo - 20, y_tobillo - 10),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
                            if lado == "izquierda":
                                angulo_tobillo_izq = angulo_tobillo
                            else:
                                angulo_tobillo_der = angulo_tobillo
                   
                            # Determinar la pierna activa
                            margen = 20
                            pierna_activa = None
                            if angulo_rodilla_izq is not None and angulo_rodilla_der is not None:
                                if abs(angulo_rodilla_izq - angulo_rodilla_der) > margen:
                                    if angulo_rodilla_izq > angulo_rodilla_der:
                                        pierna_activa = "izquierda"
                                    else:
                                        pierna_activa = "derecha"
                                else:
                                    pierna_activa = "iguales"
                                    
                            
                            # Feedback Ejercicio 1  
                            # Pierna izquierda
                            if (etapa_audio == 1 and pierna_activa == "derecha" and
                                angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                time.time() - inicio_etapa_1 >= 5 and  # aquí controlamos los 5 segundos
                                time.time() - ultimo_alerta[0] > intervalo_alerta[0] and 
                                conteo_alertas[0] < max_alertas[0]):
                                cola_voz.put("Levante la pierna izquierda.")
                                mensaje = "Levante la pierna izquierda"
                                ultimo_alerta[0] = time.time()
                                conteo_alertas[0] += 1

                                # Guardar en CSV 
                                tiempo_actual = time.time() - start
                                ejercicio_actual = f"Ejercicio 1"
                            
                                writer.writerow([
                                    round(tiempo_actual, 2),
                                    ejercicio_actual,
                                    round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                    round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                    round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                    round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                    mensaje
                                ])
    
                            # Pierna derecha
                            if (etapa_audio == 2 and pierna_activa == "izquierda" and
                                angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                time.time() - inicio_etapa_2 >= 5 and  # aquí controlamos los 5 segundos
                                time.time() - ultimo_alerta[1] > intervalo_alerta[1] and 
                                conteo_alertas[1] < max_alertas[1]):
                                cola_voz.put("Levante la pierna derecha.")
                                mensaje = "Levante la pierna derecha"
                                ultimo_alerta[1] = time.time()
                                conteo_alertas[1] += 1     

                                tiempo_actual = time.time() - start
                                ejercicio_actual = f"Ejercicio 1"
                            
                                writer.writerow([
                                    round(tiempo_actual, 2),
                                    ejercicio_actual,
                                    round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                    round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                    round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                    round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                    mensaje
                                ])
    
                            # Feedback Ejercicio 2
                            # Pierna izquierda
                            if etapa_audio == 3:
                                if angulo_rodilla_izq is not None and angulo_rodilla_der is not None: 
                                    ambos_bajo_120 = angulo_rodilla_izq < 120 and angulo_rodilla_der < 120
                                    alguno_arriba_120 = angulo_rodilla_izq >= 120 or angulo_rodilla_der >= 120
                                                        
                                if (pierna_activa == "derecha" and
                                    angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                    time.time() - ultimo_alerta[2] > intervalo_alerta[2] and 
                                    conteo_alertas[2] < max_alertas[2]):
                                    mensaje = "Levante la pierna izquierda."
                                    cola_voz.put(mensaje)
                                    ultimo_alerta[2] = time.time()
                                    conteo_alertas[2] += 1

                                    tiempo_actual = time.time() - start
                                    ejercicio_actual = f"Ejercicio 2"
                                
                                    writer.writerow([
                                        round(tiempo_actual, 2),
                                        ejercicio_actual,
                                        round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                        round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                        round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                        round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                        mensaje
                                    ])
                                
                                if (ambos_bajo_120 and
                                    time.time() - ultimo_alerta[2] > intervalo_alerta[2] and 
                                    conteo_alertas[2] < max_alertas[2]):
                                        mensaje = "Suba un poco más la pierna"
                                        cola_voz.put(mensaje)
                                        ultimo_alerta[2] = time.time()
                                        conteo_alertas[2] += 1

                                        tiempo_actual = time.time() - start
                                        ejercicio_actual = f"Ejercicio 2"
                                    
                                        writer.writerow([
                                            round(tiempo_actual, 2),
                                            ejercicio_actual,
                                            round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                            round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                            round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                            round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                            mensaje
                                        ])
                                
                                elif (alguno_arriba_120 and not mensaje_aguante_reproducido_i and time.time() - inicio_etapa_3 >= 15):
                                    mensaje = "¡Aguante un poco más!"
                                    cola_voz.put(mensaje)
                                    mensaje_aguante_reproducido_i = True

                                    tiempo_actual = time.time() - start
                                    ejercicio_actual = f"Ejercicio 2"
                                
                                    writer.writerow([
                                        round(tiempo_actual, 2),
                                        ejercicio_actual,
                                        round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                        round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                        round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                        round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                        mensaje
                                    ])
    
                            # Pierna derecha
                            if etapa_audio == 4:
                                if angulo_rodilla_izq is not None and angulo_rodilla_der is not None: 
                                    ambos_bajo_120 = angulo_rodilla_izq < 120 and angulo_rodilla_der < 120
                                    alguno_arriba_120 = angulo_rodilla_izq >= 120 or angulo_rodilla_der >= 120
                                                        
                                if (pierna_activa == "izquierda" and
                                    angulo_rodilla_izq is not None and angulo_rodilla_der is not None and
                                    time.time() - ultimo_alerta[3] > intervalo_alerta[3] and 
                                    conteo_alertas[3] < max_alertas[3]):
                                    mensaje = "Levante la pierna derecha."
                                    cola_voz.put(mensaje)
                                    ultimo_alerta[3] = time.time()
                                    conteo_alertas[3] += 1

                                    tiempo_actual = time.time() - start
                                    ejercicio_actual = f"Ejercicio 2"
                                
                                    writer.writerow([
                                        round(tiempo_actual, 2),
                                        ejercicio_actual,
                                        round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                        round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                        round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                        round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                        mensaje
                                    ])
                                
                                if (ambos_bajo_120 and
                                    time.time() - ultimo_alerta[3] > intervalo_alerta[3] and 
                                    conteo_alertas[3] < max_alertas[3]):
                                        mensaje = "Suba un poco más la pierna"
                                        cola_voz.put(mensaje)
                                        ultimo_alerta[3] = time.time()
                                        conteo_alertas[3] += 1

                                        tiempo_actual = time.time() - start
                                        ejercicio_actual = f"Ejercicio 2"
                                    
                                        writer.writerow([
                                            round(tiempo_actual, 2),
                                            ejercicio_actual,
                                            round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                            round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                            round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                            round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                            mensaje
                                        ])
                                
                                elif (alguno_arriba_120 and not mensaje_aguante_reproducido_d and time.time() - inicio_etapa_4 >= 15):
                                    mensaje = "¡Aguante un poco más!"
                                    cola_voz.put(mensaje)
                                    mensaje_aguante_reproducido_d = True

                                    tiempo_actual = time.time() - start
                                    ejercicio_actual = f"Ejercicio 2"
                                
                                    writer.writerow([
                                        round(tiempo_actual, 2),
                                        ejercicio_actual,
                                        round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                        round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                        round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                        round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                        mensaje
                                    ])
    
                            # Feedback ejercicio 3
                            # Pierna izquierda
                            elif etapa_audio == 5:
                                if angulo_rodilla_izq is not None and angulo_rodilla_der is not None:
                                    ambas_bajo_110 = angulo_rodilla_izq < 110 and angulo_rodilla_der < 110
                                    alguna_arriba_110 = angulo_rodilla_izq >= 110 or angulo_rodilla_der >= 110

                                if ambas_bajo_110:
                                    if (time.time() - ultimo_alerta[4] > intervalo_alerta[4] and 
                                    conteo_alertas[4] < max_alertas[4]):
                                        mensaje = "No baje la pierna"
                                        cola_voz.put(mensaje)
                                        ultimo_alerta[4] = time.time()
                                        conteo_alertas[4] += 1

                                        tiempo_actual = time.time() - start
                                        ejercicio_actual = f"Ejercicio 3"
                                    
                                        writer.writerow([
                                            round(tiempo_actual, 2),
                                            ejercicio_actual,
                                            round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                            round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                            round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                            round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                            mensaje
                                        ])
                        
                                elif (alguna_arriba_110 and not mensaje_siga_reproducido_i and time.time() - inicio_etapa_5 >= 15):  # Aquí controlamos el retraso de 15s
                                    cola_voz.put("¡Siga así, ya casi está!")
                                    mensaje_siga_reproducido_i = True

                                    tiempo_actual = time.time() - start
                                    ejercicio_actual = f"Ejercicio 3"
                                
                                    writer.writerow([
                                        round(tiempo_actual, 2),
                                        ejercicio_actual,
                                        round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                        round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                        round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                        round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                        mensaje
                                    ])
    
                            # Pierna derecha
                            elif etapa_audio == 6:
                                if angulo_rodilla_izq is not None and angulo_rodilla_der is not None:
                                    ambas_bajo_110 = angulo_rodilla_izq < 110 and angulo_rodilla_der < 110
                                    alguna_arriba_110 = angulo_rodilla_izq >= 110 or angulo_rodilla_der >= 110

                            
                                if ambas_bajo_110:
                                    if (time.time() - ultimo_alerta[5] > intervalo_alerta[5] and 
                                    conteo_alertas[5] < max_alertas[5]):
                                        mensaje = "No baje la pierna"
                                        cola_voz.put(mensaje)
                                        ultimo_alerta[5] = time.time()
                                        conteo_alertas[5] += 1

                                        tiempo_actual = time.time() - start
                                        ejercicio_actual = f"Ejercicio 3"
                                    
                                        writer.writerow([
                                            round(tiempo_actual, 2),
                                            ejercicio_actual,
                                            round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                            round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                            round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                            round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                            mensaje
                                        ])
                                        
                        
                                elif (alguna_arriba_110 and not mensaje_siga_reproducido_d and 
                                  time.time() - inicio_etapa_6 >= 15):  # Aquí controlamos el retraso de 15s
                                    cola_voz.put("¡Siga así, ya casi está!")
                                    mensaje_siga_reproducido_d = True

                                    tiempo_actual = time.time() - start
                                    ejercicio_actual = f"Ejercicio 3"
                                
                                    writer.writerow([
                                        round(tiempo_actual, 2),
                                        ejercicio_actual,
                                        round(angulo_rodilla_izq, 2) if angulo_rodilla_izq is not None else None,
                                        round(angulo_rodilla_der, 2) if angulo_rodilla_der is not None else None,
                                        round(angulo_tobillo_izq, 2) if angulo_tobillo_izq is not None else None,
                                        round(angulo_tobillo_der, 2) if angulo_tobillo_der is not None else None,
                                        mensaje
                                    ])
                        
                        except:
                            pass
                                
    
    
                # Guardar frame y mostrarlo en la vetana
                out.write(annotated)
                cv2.imshow('Pose en tiempo real', annotated)
              
                key = cv2.waitKey(1) & 0xFF
                if key == 32 or key == ord('q'):
                    print("Grabación detenida por el usuario.")
                    break

    stop_audio.set()
    cap.release() 
    out.release()
    cv2.destroyAllWindows()
    print(f"Video guardado en: {filename}")

# Ejecutar
grabar_con_pose()

Presiona Enter para iniciar la grabación.


Grabación iniciada. Pulsa 'espacio' o 'q' para detener.
Grabación detenida por el usuario.
Video guardado en: video_full_43.avi


In [14]:
import os

# Ruta del archivo CSV (asegúrate de actualizarla según corresponda)
output_path = os.path.join(os.path.expanduser("~"), "Escritorio", "VIDEOS")
csv_path = os.path.join(output_path, "datos_pose.csv")

# Mostrar contenido del archivo CSV
try:
    with open(csv_path, mode='r') as file:
        reader = csv.reader(file)
        for row in reader:
            print(row)
except FileNotFoundError:
    print(f"No se encontró el archivo: {csv_path}")

['Tiempo', 'Ejercicio', 'Ángulo Rodilla Izquierda', 'Ángulo Rodilla Derecha', 'Ángulo Tobillo Izquierdo', 'Ángulo Tobillo Derecho', 'Detalle Alerta']
['26.19', 'Ejercicio 1', '99.82', '126.09', '148.75', '143.07', 'Levante la pierna izquierda']
['40.06', 'Ejercicio 1', '122.46', '97.27', '135.64', '150.34', 'Levante la pierna derecha']
['62.64', 'Ejercicio 2', '101.12', '113.8', '150.15', '129.4', 'Suba un poco más la pierna']
['77.64', 'Ejercicio 2', '176.6', '99.05', '112.05', '148.95', '¡Aguante un poco más!']
['81.31', 'Ejercicio 2', '101.74', '96.31', '148.85', '145.81', 'Suba un poco más la pierna']
['91.37', 'Ejercicio 2', '105.72', '167.01', '145.78', '101.34', 'Levante la pierna izquierda.']
['92.96', 'Ejercicio 2', '106.56', '98.98', '140.1', '141.93', 'Suba un poco más la pierna']
['108.02', 'Ejercicio 2', '97.32', '172.89', '144.03', '96.43', '¡Aguante un poco más!']
['110.11', 'Ejercicio 2', '95.42', '114.06', '144.65', '138.44', 'Suba un poco más la pierna']
['122.43', 'E